# Vertex AI Search : Streaming (StreamAnswer)

### Install client library

Install the following packages required to execute this notebook.

In [1]:
%pip install --upgrade --quiet google-cloud-discoveryengine

Note: you may need to restart the kernel to use updated packages.


In [24]:
import os
from dotenv import load_dotenv

load_dotenv()
PROJECT_ID = os.environ.get("PROJECT_ID")
LOCATION = os.environ.get("REGION", "global")
ENGINE_ID = os.environ.get("ENGINE_ID")

### Import libraries

In [4]:
from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine_v1 as discoveryengine
from google.protobuf.json_format import MessageToDict

## Search method

Use the [`search()`](https://cloud.google.com/agentspace/agentspace-enterprise/docs/preview-search-results) method to retrieve all search results for a given query.

In [ ]:
def get_search_sdk_response(
    project_id: str,
    location: str,
    engine_id: str,
    search_query: str,
    prompt_preamble: str | None = None,
) -> list[discoveryengine.SearchResponse]:
 
    client_options = (
        ClientOptions(api_endpoint=f"{location}-discoveryengine.googleapis.com")
        if location != "global"
        else None
    )

    client = discoveryengine.SearchServiceClient(client_options=client_options)

    serving_config = f"projects/{project_id}/locations/{location}/collections/default_collection/engines/{engine_id}/servingConfigs/default_config"

    content_search_spec = discoveryengine.SearchRequest.ContentSearchSpec(
        snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
            return_snippet=True
        ),
        summary_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec(
            summary_result_count=5,
            include_citations=True,
            ignore_adversarial_query=True,
            ignore_non_summary_seeking_query=True,
            model_prompt_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec.ModelPromptSpec(
                preamble=prompt_preamble
            ),
            model_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec.ModelSpec(
                version="stable",
            ),
        ),
        # -------extractive content-----
        #extractive_content_spec=discoveryengine.SearchRequest.ContentSearchSpec.ExtractiveContentSpec(
        #    max_extractive_segment_count=2,
        #    max_extractive_answer_count=2,
        #    return_extractive_segment_score=True,
        #),
    )

    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=search_query,
        page_size=10,
        content_search_spec=content_search_spec,
        query_expansion_spec=discoveryengine.SearchRequest.QueryExpansionSpec(
            condition=discoveryengine.SearchRequest.QueryExpansionSpec.Condition.AUTO,
        ),
        spell_correction_spec=discoveryengine.SearchRequest.SpellCorrectionSpec(
            mode=discoveryengine.SearchRequest.SpellCorrectionSpec.Mode.AUTO
        ),
    )

    response = client.search(request)
    return response

In [ ]:
search_query = "탄소 중립에 대해 설명해줘"

response = get_search_sdk_response(
    PROJECT_ID,
    LOCATION,
    ENGINE_ID,
    search_query,
)

print(response.summary)

## Answer / Streaming Answer Method

Use the [`answer()`](https://cloud.google.com/agentspace/agentspace-enterprise/docs/answer) method to provide a conversational experience for search results.

* Answer : answer_query
* Streaming Answer : stream_answer_query

In [55]:
def get_answer_sdk_response(
    project_id: str,
    location: str,
    engine_id: str,
    query: str,
) -> discoveryengine.AnswerQueryResponse:
    
    client_options = (
        ClientOptions(api_endpoint=f"{location}-discoveryengine.googleapis.com")
        if location != "global"
        else None
    )

    client = discoveryengine.ConversationalSearchServiceClient(
        client_options=client_options
    )

    serving_config = f"projects/{project_id}/locations/{location}/collections/default_collection/engines/{engine_id}/servingConfigs/default_serving_config"

    query_understanding_spec = discoveryengine.AnswerQueryRequest.QueryUnderstandingSpec(
        query_rephraser_spec=discoveryengine.AnswerQueryRequest.QueryUnderstandingSpec.QueryRephraserSpec(
            disable=False,  # Optional: Disable query rephraser
            max_rephrase_steps=1,  # Optional: Number of rephrase steps
        ),
        # Optional: Classify query types
        query_classification_spec=discoveryengine.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec(
            types=[
                discoveryengine.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec.Type.ADVERSARIAL_QUERY,
                discoveryengine.AnswerQueryRequest.QueryUnderstandingSpec.QueryClassificationSpec.Type.NON_ANSWER_SEEKING_QUERY,
            ]  # Options: ADVERSARIAL_QUERY, NON_ANSWER_SEEKING_QUERY or both
        ),
    )
 
    answer_generation_spec = discoveryengine.AnswerQueryRequest.AnswerGenerationSpec(
        ignore_adversarial_query=False,  # Optional: Ignore adversarial query
        ignore_non_answer_seeking_query=False,  # Optional: Ignore non-answer seeking query
        ignore_low_relevant_content=False,  # Optional: Return fallback answer when content is not relevant
        model_spec=discoveryengine.AnswerQueryRequest.AnswerGenerationSpec.ModelSpec(
            model_version="stable",  # Optional: Model to use for answer generation
        ),
        prompt_spec=discoveryengine.AnswerQueryRequest.AnswerGenerationSpec.PromptSpec(
            preamble=None,  # Optional: Natural language instructions for customizing the answer.
        ),
        include_citations=True,  # Optional: Include citations in the response
        answer_language_code="ko",  # Optional: Language code of the answer
    )

    # Initialize request argument(s)
    request = discoveryengine.AnswerQueryRequest(
        serving_config=serving_config,
        query=discoveryengine.Query(text=query),
        session=None,  # Optional: include previous session ID to continue a conversation
        query_understanding_spec=query_understanding_spec,
        answer_generation_spec=answer_generation_spec,
    )

    # Make the request
    #response = client.answer_query(request)

    # Streaming Answer
    stream = client.stream_answer_query(request=request)
    for response in stream:
        #print(response)
        print(response.answer.references)

    return response

In [ ]:
query = "탄소 중립에 대해 설명해줘"
response = get_answer_sdk_response(
    PROJECT_ID,
    LOCATION,
    ENGINE_ID,
    query,
)

print(response.answer.answer_text)